# 01. Environment and GPU

## 学習目標

- 学習に使うハードウェア（GPU・VRAM・BF16対応・SDPAバックエンド）を**実測で**把握する
- 「なぜ BF16 か」「なぜ SDPA か」を説明できるようになる
- ハードウェアから初期設定（micro batch / grad accum / モデルサイズ）を導く**根拠**を知る

## 前提知識

浮動小数点数が「符号・指数・仮数」で表現されること。

In [1]:
# 共通セットアップ（全ノートブック同一）
import warnings
warnings.filterwarnings("ignore")

import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT={ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}")

ROOT=/home/kazumasa/projects/jp_llm_lab
device=cuda, torch=2.11.0+cu128


In [2]:
from dataclasses import asdict
from jp_llm_lab.utils.env import collect_env_report, recommend_setup

report = collect_env_report()
pd.Series(asdict(report)).to_frame("value")

,value
python_version,3.12.3
torch_version,2.11.0+cu128
cuda_version,12.8
cuda_available,True
gpu_name,NVIDIA GeForce RTX 5080
vram_gb,15.92
capability,12.0
bf16_supported,True
sdpa_backends,"[math, flash, mem_efficient, cudnn]"
compile_available,True


## 数値形式: なぜ BF16 か

| 形式 | 符号 | 指数 | 仮数 | 表現範囲 | 相対精度 |
|---|---|---|---|---|---|
| FP32 | 1 | 8 | 23 | ~1e±38 | ~7桁 |
| **BF16** | 1 | **8** | 7 | **~1e±38（FP32と同じ）** | ~2-3桁 |
| FP16 | 1 | 5 | 10 | ~6e±4 | ~3桁 |

- BF16 は**指数部が FP32 と同じ8bit** → 勾配のダイナミックレンジがそのまま収まり、
  FP16 で必要な loss scaling が不要。
- 仮数が7bitしかない粗さは、勾配ノイズ（ミニバッチ由来）より小さいオーダーであることが多く、
  学習品質への影響は限定的（これは M3 で fp32 と実測比較する）。
- メモリ半減・Tensor Core の高速パスが使える。

In [3]:
setup = recommend_setup(report)
pd.Series(setup.as_dict()).to_frame("recommendation")

,recommendation
device,cuda
dtype,bf16
attn_impl,sdpa
model_recommendation,L
micro_batch,"{'S': 64, 'M': 32, 'L': 16}"
grad_accum,"{'S': 1, 'M': 1, 'L': 2}"
effective_tokens_target,16384
notes,[torch.compile available — benchmark on/off in...


### 推奨ロジック（`recommend_setup` の中身）

1. **dtype**: CUDA + BF16対応 → `bf16`、それ以外は `fp32`
2. **attn_impl**: 学習は SDPA（fused kernel）。教育・分析用に explicit 実装が常に併存（出力一致はテスト済み）
3. **micro_batch**: VRAM 段階表（<6GB / <12GB / ≥12GB）。活性メモリは概ね `n_layers·T·d_model` に比例するためモデルサイズ別
4. **grad_accum**: `ceil(目標トークン数 ÷ (micro_batch × context_len))` — マシンが違っても**1ステップあたりのトークン数**を揃える

**Caveat**: これは粗い事前推定。実測最適点は M3 §14.1 のハードウェア校正で決める。

In [4]:
# 行列積のマイクロベンチ: fp32 vs bf16（Tensor Coreの効果を体感する）
def matmul_tflops(dtype, n=4096, reps=10):
    if DEVICE == "cpu":
        n, reps = 1024, 3
    a = torch.randn(n, n, device=DEVICE, dtype=dtype)
    b = torch.randn(n, n, device=DEVICE, dtype=dtype)
    for _ in range(3):
        a @ b  # warmup
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        a @ b
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / reps
    return 2 * n**3 / dt / 1e12

rows = [("fp32", matmul_tflops(torch.float32))]
if DEVICE == "cuda":
    rows.append(("bf16", matmul_tflops(torch.bfloat16)))
df = pd.DataFrame(rows, columns=["dtype", "TFLOPs"])
if len(rows) == 2:
    print(f"bf16 は fp32 の ×{rows[1][1]/rows[0][1]:.1f}（4096×4096 行列積, 中央値でなく平均）")
fig = go.Figure(go.Bar(x=df["dtype"], y=df["TFLOPs"], marker_color=["#1f77b4", "#2ca02c"][:len(df)]))
fig.update_layout(title="行列積スループット（大きいほど速い）", yaxis_title="TFLOPs",
                  template="plotly_white", height=360)
fig.show()
df

bf16 は fp32 の ×2.7（4096×4096 行列積, 中央値でなく平均）


,dtype,TFLOPs
0,fp32,14.995792
1,bf16,41.226580


## 読み方と注意

- **Observation**: 上のセルの実測値がそのまま観察結果（このノートブックは実行のたびに再計測する）。
  RTX 5080 では bf16 が fp32 の数倍になるはず。
- **Interpretation**: 差は Tensor Core の bf16 パスによる。ただし**学習全体がこの倍率で速くなるわけではない**
  — 小さいモデルではカーネル起動・データ供給・評価がボトルネックになる（NB13で実測）。
- **Caveat**: この数値は行列サイズ・電力状態・ドライバに依存する。「GPUの公称TFLOPs」との差は
  実行効率であり異常ではない。

## 確認問題

1. FP16 ではなく BF16 を使う根拠を「指数部」という語を使って説明せよ。
2. `grad_accum` を増やしたとき、1 optimizer step あたりのトークン数はどうなるか。
3. VRAM が半分のマシンで同じ実験を再現するには、何を変えて何を固定すべきか。

**次へ**: [03_tokenizer_anatomy](03_tokenizer_anatomy.ipynb)（NB02 コーパス探索は M2 で追加）